# 서울시 따릉이 데이터 전처리 (ISLP 모델링 목적 - 원본 이력 보존형)

이 노트북은 서울시 공공자전거(따릉이) 데이터와 기상/미세먼지 데이터를 결합하여 ISLP 다중선형회귀 및 교호작용 분석에 활용할 수 있도록 전처리하는 파이프라인입니다.

**[원본 대여 이력 데이터 보존 최적화]**
- 일 단위로 대여건수를 무작정 합산(GroupBy)하지 않고, 개별 사용자의 **출발 시간대, 도착지, 이용 시간 및 거리** 등의 원본 이력을 모두 그대로 유지합니다.
- 우리가 타겟하는 대여소(Office, Leisure, University)에서 출발한 이력만을 필터링해 내어 대용량 CSV 파일들의 크기를 최적화합니다.
- 각 대여 이력(시간대별/이동별 Row)에 해당 일자의 기온/강수량/미세먼지 데이터를 복사하여 붙여넣음(Left Join)으로써 완벽한 통합 분석 테이블을 구성합니다.

## 1. 라이브러리 및 경로 설정

In [20]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

WEATHER_DIR = os.path.join(DATA_DIR, '기상정보(03.31_05.21)')
BIKE_DIR = os.path.join(DATA_DIR, '따릉이(03,04,05.01_05.15)')
MASTER_DIR = os.path.join(DATA_DIR, '따릉이_마스터정보(대여소_id)')
DUST_DIR = os.path.join(DATA_DIR, '미세먼지 정보(03.31_05.21)')

START_DATE = '2026-03-01'
END_DATE = '2026-05-15'

def find_encoding(file_path):
    for enc in ['utf-8', 'cp949', 'euc-kr']:
        try:
            pd.read_csv(file_path, encoding=enc, nrows=1)
            return enc
        except UnicodeDecodeError:
            continue
    return 'cp949'

## 2. 마스터 데이터 기반 타겟 대여소 추출 (University 확장)

In [21]:
def load_master_data():
    master_files = glob.glob(os.path.join(MASTER_DIR, '*.csv'))
    if not master_files:
        raise FileNotFoundError("마스터 데이터 폴더에 CSV 파일이 없습니다.")
    
    master_file = master_files[0]
    enc = find_encoding(master_file)
    master_df = pd.read_csv(master_file, encoding=enc)
    
    station_name_col = next((col for col in master_df.columns if '명' in col or '이름' in col), master_df.columns[2])
    station_id_col = next((col for col in master_df.columns if 'ID' in col.upper() or '번호' in col), master_df.columns[0])
    
    office_keywords = ['여의도역', '강남역', '광화문역', '을지로입구역', '시청역', '선릉역', '삼성역', '종각역', '가산디지털단지역', '판교역', '공덕역']
    leisure_keywords = ['여의나루역', '뚝섬유원지역', '서울숲역', '반포한강공원', '망원한강공원', '올림픽공원', '월드컵경기장', '어린이대공원', '노들섬', '석촌호수']
    university_keywords = ['공릉', '서울과학기술대학교']

    def assign_station_type(name):
        name = str(name)
        for kw in office_keywords:
            if kw in name: return 'Office'
        for kw in leisure_keywords:
            if kw in name: return 'Leisure'
        for kw in university_keywords:
            if kw in name: return 'University'
        return None

    master_df['station_type'] = master_df[station_name_col].apply(assign_station_type)
    target_master_df = master_df.dropna(subset=['station_type']).copy()
    
    target_master_df = target_master_df[[station_id_col, station_name_col, 'station_type']]
    target_master_df.columns = ['station_id', 'station_name', 'station_type']
    
    return target_master_df

target_master_df = load_master_data()
print(f"{len(target_master_df)}개의 타겟 대여소(Office/Leisure/University) 식별 완료.")
display(target_master_df.head())

31개의 타겟 대여소(Office/Leisure/University) 식별 완료.


,station_id,station_name,station_type
10,ST-99,뚝섬유원지역 1번출구 앞,Leisure
48,ST-955,삼성역 5~6번 출구,Office
143,ST-87,홈플러스 월드컵경기장점,Leisure
214,ST-805,삼성역 3번 출구,Office
219,ST-800,삼성역 7번출구,Office


## 3. 대규모 따릉이 이력 원본 추출 (메모리 최적화)
시간대, 도착지, 거리, 시간 등의 원본 컬럼을 전혀 훼손하지 않고 그대로 추출해 옵니다.

In [22]:
def load_and_filter_raw_bike_data(folder_path, target_master_df):
    all_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
    df_list = []
    
    target_station_ids = target_master_df['station_id'].unique()

    for file in all_files:
        enc = find_encoding(file)
        df = pd.read_csv(file, encoding=enc)
        
        start_id_cols = [c for c in df.columns if '시작' in c and 'ID' in c.upper()]
        if not start_id_cols:
            start_id_cols = [c for c in df.columns if '대여소_ID' in c.upper() or '대여대여소' in c]
            
        if not start_id_cols: continue
            
        join_col = start_id_cols[0]
        
        filtered_df = df[df[join_col].isin(target_station_ids)].copy()
        if filtered_df.empty: continue
            
        filtered_df = pd.merge(filtered_df, target_master_df[['station_id', 'station_type']], left_on=join_col, right_on='station_id', how='inner')
        filtered_df.drop(columns=['station_id'], inplace=True)
        
        date_col = next((c for c in filtered_df.columns if '날짜' in c or '일자' in c), filtered_df.columns[0])
        filtered_df['date'] = pd.to_datetime(filtered_df[date_col].astype(str).str.replace('-', '')).dt.normalize()
        
        df_list.append(filtered_df)
            
    if not df_list:
        return pd.DataFrame()
        
    merged_bike_df = pd.concat(df_list, ignore_index=True)
    return merged_bike_df

bike_raw_df = load_and_filter_raw_bike_data(BIKE_DIR, target_master_df)
print(f"필터링된 대여 이력 원본 데이터 병합 완료: 총 {len(bike_raw_df):,} 건")
display(bike_raw_df.head())

필터링된 대여 이력 원본 데이터 병합 완료: 총 232,648 건


,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,station_type,date
0,20260508,출발시간,1540,ST-1586,잠실6동_001_1,ST-811,역삼1동_009_1,4,195.0,30067.0,Leisure,2026-05-08
1,20260508,출발시간,1400,ST-73,여의동_010_1,ST-3401,여의동_006_9,1,91.0,7210.0,Leisure,2026-05-08
2,20260508,출발시간,1740,ST-99,자양3동_036_1,ST-1335,면목2동_026_1,1,62.0,12552.0,Leisure,2026-05-08
3,20260508,출발시간,1925,ST-1720,오륜동_001_4,ST-505,성내3동_012_1,1,13.0,2080.0,Leisure,2026-05-08
4,20260508,출발시간,1655,ST-73,여의동_010_1,ST-2491,가양1동_001_3,1,52.0,11009.0,Leisure,2026-05-08


## 4. 기상 및 미세먼지 환경 데이터 로드 및 시계열 보간
기준 날짜 범위를 완전하게 덮도록 보강(Reindex)한 후 시계열 보간을 수행합니다.

In [23]:
def load_environment_data(folder_path, value_col_hints, output_col_name):
    all_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
    df_list = []
    
    for file in all_files:
        enc = find_encoding(file)
        df = pd.read_csv(file, encoding=enc)
        
        date_col = next((c for c in df.columns if '일시' in c or '일자' in c or '날짜' in c), df.columns[0])
        val_cols = []
        for hint in value_col_hints:
            found = next((c for c in df.columns if hint in c), None)
            if found: val_cols.append(found)
            
        if val_cols:
            sub_df = df[[date_col] + val_cols].copy()
            sub_df['date'] = pd.to_datetime(sub_df[date_col]).dt.normalize()
            sub_df = sub_df.drop(columns=[date_col])
            df_list.append(sub_df)
            
    if not df_list:
        return pd.DataFrame()
        
    merged_env = pd.concat(df_list, ignore_index=True)
    merged_env = merged_env.groupby('date').mean().reset_index()
    
    rename_dict = {}
    for hint, out_name in zip(value_col_hints, output_col_name):
        found = next((c for c in merged_env.columns if hint in c), None)
        if found:
            rename_dict[found] = out_name
            
    merged_env = merged_env.rename(columns=rename_dict)
    return merged_env

weather_df = load_environment_data(WEATHER_DIR, ['기온', '강수'], ['temp', 'precip'])
dust_df = load_environment_data(DUST_DIR, ['미세먼지', 'PM10'], ['pm10', 'pm10'])

if not weather_df.empty and not dust_df.empty:
    env_df = pd.merge(weather_df, dust_df, on='date', how='outer')
else:
    env_df = weather_df if not weather_df.empty else dust_df

env_df.set_index('date', inplace=True)
env_df.sort_index(inplace=True)

# 데이터가 비어있을 수 있는 앞뒤 날짜를 모두 덮도록 Reindex
target_dates = pd.date_range(start=START_DATE, end=END_DATE)
env_df = env_df.reindex(target_dates)

if 'precip' in env_df.columns:
    env_df['precip'] = env_df['precip'].fillna(0)
if 'temp' in env_df.columns:
    env_df['temp'] = env_df['temp'].interpolate(method='time').bfill().ffill()
if 'pm10' in env_df.columns:
    env_df['pm10'] = env_df['pm10'].interpolate(method='time').bfill().ffill()
    
env_df.reset_index(inplace=True)
env_df.rename(columns={'index': 'date'}, inplace=True)
print("환경 데이터 보간 처리 완료.")
display(env_df.head())

환경 데이터 보간 처리 완료.


,date,temp,precip,pm10
0,2026-03-01,9.1,0.53,23.178571
1,2026-03-02,4.9,17.25,11.238095
2,2026-03-03,5.9,4.92,11.333333
3,2026-03-04,6.0,0.00,21.760000
4,2026-03-05,6.0,6.50,26.714286


## 5. 전체 데이터 최종 결합(Left Join) 및 파생변수 생성
수많은 대여 이력들(수십만 행) 각각에 대하여 동일한 일자(`date`)의 기온, 강수량, 미세먼지가 조인(Left Join)됩니다.

In [24]:
final_df = pd.merge(bike_raw_df, env_df, on='date', how='left')

# 분석 기간 필터링
mask = (final_df['date'] >= pd.to_datetime(START_DATE)) & (final_df['date'] <= pd.to_datetime(END_DATE))
final_df = final_df.loc[mask].copy()

# 주말 파생변수
final_df['is_weekend'] = final_df['date'].dt.dayofweek.isin([5, 6]).astype(int)

# [ISLP 최적화] statsmodels 더미화를 위한 범주형(Category) 변환
final_df['station_type'] = final_df['station_type'].astype('category')
final_df['is_weekend'] = final_df['is_weekend'].astype('category')

# date 컬럼을 맨 앞으로
cols = final_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('date')))
final_df = final_df[cols]

print("[최종 데이터프레임 구조 확인]")
final_df.info()

[최종 데이터프레임 구조 확인]
<class 'pandas.core.frame.DataFrame'>
Index: 232648 entries, 0 to 232647
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   date          232648 non-null  datetime64[ns]
 1   기준_날짜         232648 non-null  int64         
 2   집계_기준         232648 non-null  object        
 3   기준_시간대        232648 non-null  int64         
 4   시작_대여소_ID     232648 non-null  object        
 5   시작_대여소명       227993 non-null  object        
 6   종료_대여소_ID     232648 non-null  object        
 7   종료_대여소명       226740 non-null  object        
 8   전체_건수         232648 non-null  int64         
 9   전체_이용_분       225968 non-null  float64       
 10  전체_이용_거리      225968 non-null  float64       
 11  station_type  232648 non-null  category      
 12  temp          232648 non-null  float64       
 13  precip        232648 non-null  float64       
 14  pm10          232648 non-null  float64       
 15  is_w

## 6. 결과 저장

In [25]:
output_path = os.path.join(BASE_DIR, 'preprocessed_data_for_islp.csv')
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"{START_DATE} ~ {END_DATE} 구간 대여 이력 전처리 완료. '{output_path}' 에 저장되었습니다.")
display(final_df.head())

2026-03-01 ~ 2026-05-15 구간 대여 이력 전처리 완료. '/Users/gyuhan/Desktop/project/데이터분석 텀프/preprocessed_data_for_islp.csv' 에 저장되었습니다.


,date,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,station_type,temp,precip,pm10,is_weekend
0,2026-05-08,20260508,출발시간,1540,ST-1586,잠실6동_001_1,ST-811,역삼1동_009_1,4,195.0,30067.0,Leisure,15.2,0.0,24.208333,0
1,2026-05-08,20260508,출발시간,1400,ST-73,여의동_010_1,ST-3401,여의동_006_9,1,91.0,7210.0,Leisure,15.2,0.0,24.208333,0
2,2026-05-08,20260508,출발시간,1740,ST-99,자양3동_036_1,ST-1335,면목2동_026_1,1,62.0,12552.0,Leisure,15.2,0.0,24.208333,0
3,2026-05-08,20260508,출발시간,1925,ST-1720,오륜동_001_4,ST-505,성내3동_012_1,1,13.0,2080.0,Leisure,15.2,0.0,24.208333,0
4,2026-05-08,20260508,출발시간,1655,ST-73,여의동_010_1,ST-2491,가양1동_001_3,1,52.0,11009.0,Leisure,15.2,0.0,24.208333,0
